In [9]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ tibble    3.2.1
✔ ggplot2   4.0.1     ✔ tidyr     1.3.0
✔ lubridate 1.9.3     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


# CPOD para todo mundo

## Criando resultados por lote

In [151]:
# dados = read.csv("/home/aninha/Desktop/Doutorado/Dados/papiron_todas_pans/lote1/parte1/results.csv")
# glimpse(dados)

In [152]:
# head(dados)

In [153]:
# unique(dados1$achado)

In [154]:
# library(dplyr)
# library(tidyr)

# calcular_cpod_por_imagem <- function(df) {
  
#   df_wide <- df %>%
#     mutate(
#       componente = case_when(
#         achado == "carie" ~ "C",                     # Cariado
#         achado == "dente ausente" ~ "P",                   # Perdido
#         achado %in% c(
#           "restauracao",
#           "condutos obturados",
#           "coroa unitaria sobre dente",
#           "protese fixa sobre dente"
#         ) ~ "O",                                     # Obturado
#         TRUE ~ NA_character_
#       )
#     ) %>%
#     # Mantém só os achados que entram no CPOD
#     filter(!is.na(componente)) %>%
#     # Um dente conta no máximo uma vez por componente
#     distinct(imagem, dente, componente) %>%
#     count(imagem, componente, name = "n") %>%
#     pivot_wider(
#       names_from = componente,
#       values_from = n,
#       values_fill = 0
#     )
  
#   # Garante que as colunas C, P e O existam
#   for (nm in c("C", "P", "O")) {
#     if (!nm %in% names(df_wide)) {
#       df_wide[[nm]] <- 0L
#     }
#   }
  
#   df_final <- df_wide %>%
#     rename(
#       cariado  = C,
#       ausente  = P,
#       obturado = O
#     ) %>%
#     mutate(
#       CPOD = cariado + ausente + obturado
#     )
  
#   return(df_final)
# }

In [10]:
library(dplyr)
library(tidyr)

calcular_cpod_por_imagem <- function(df) {
  
  df_wide <- df %>%
    mutate(
      componente = case_when(
        achado == "carie" ~ "C",                   
        achado == "dente ausente" ~ "P",           
        achado %in% c(
          "restauracao",
          "condutos obturados",
          "coroa unitaria sobre dente",
          "protese fixa sobre dente"
        ) ~ "O",                                    
        TRUE ~ NA_character_
      )
    ) %>%
    # mantém só C, P, O
    filter(!is.na(componente)) %>%
    
    # Garante explicitamente que cada dente entra no máximo uma vez por componente
    group_by(imagem, dente, componente) %>%
    summarise(flag = 1, .groups = "drop") %>%
    
    # Conta quantos dentes têm cada componente
    count(imagem, componente, name = "n") %>%
    
    # transforma C/P/O em colunas
    pivot_wider(
      names_from = componente,
      values_from = n,
      values_fill = 0
    )
  
  # Garante existência das colunas (caso alguma imagem não tenha C ou P ou O)
  for (nm in c("C", "P", "O")) {
    if (!nm %in% names(df_wide)) {
      df_wide[[nm]] <- 0L
    }
  }
  
  df_final <- df_wide %>%
    rename(
      cariado  = C,
      ausente  = P,
      obturado = O
    ) %>%
    mutate(
      CPOD = cariado + ausente + obturado
    )
  
  return(df_final)
}


In [11]:
# cpod_img <- calcular_cpod_por_imagem(dados)
# head(cpod_img)

In [12]:
# write.csv(cpod_img, "/home/aninha/Desktop/Doutorado/Dados/papiron_todas_pans/cpod_9_dez/cpod_resultado_sem_rep1.csv", row.names = FALSE)

In [13]:
# stop("Interrompendo o script.")

## CPOD final para todos

In [14]:
library(dplyr)
library(purrr)
library(readr)

# Diretório onde estão os arquivos
dir_path <- "/home/aninha/Desktop/Doutorado/Dados/papiron_todas_pans/cpod_9_dez/"

# Lista todos os arquivos resultado1,2,3,4 especificamente
arquivos <- list.files(dir_path, full.names = TRUE) %>% 
  grep("cpod_resultado_sem_rep[1-4]\\.csv$", ., value = TRUE)

dados <- map_dfr(arquivos, read_csv)

glimpse(dados)

write.csv(dados, "/home/aninha/Desktop/Doutorado/Dados/papiron_todas_pans/cpod_9_dez/cpod_resultado_sem_rep_final.csv", row.names = FALSE)

Rows: 49135 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): imagem
dbl (4): obturado, ausente, cariado, CPOD

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 49189 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): imagem
dbl (4): obturado, ausente, cariado, CPOD

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 49210 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): imagem
dbl (4): obturado, ausente, cariado, CPOD

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows

Rows: 196,670
Columns: 5
$ imagem   <chr> "ALP.025868-pan.jpg", "ALP.025874-pan.jpg", "ALP.025905-pan.j…
$ obturado <dbl> 10, 5, 7, 11, 3, 2, 5, 0, 11, 1, 11, 7, 6, 7, 15, 0, 13, 3, 3…
$ ausente  <dbl> 8, 15, 11, 5, 3, 0, 0, 4, 2, 0, 1, 0, 0, 0, 3, 2, 14, 3, 2, 8…
$ cariado  <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0…
$ CPOD     <dbl> 18, 20, 18, 16, 6, 2, 5, 4, 13, 1, 12, 7, 7, 7, 18, 2, 27, 6,…


In [15]:
head(dados)

imagem,obturado,ausente,cariado,CPOD
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
ALP.025868-pan.jpg,10,8,0,18
ALP.025874-pan.jpg,5,15,0,20
ALP.025905-pan.jpg,7,11,0,18
ALP.025906-pan.jpg,11,5,0,16
ALP.025920-pan.jpg,3,3,0,6
ALP.025921-pan.jpg,2,0,0,2


In [16]:
summary(dados$CPOD)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
      1       6      11      12      18      33 

In [17]:
library(stringr)

dados <- dados %>%
  mutate(
    image_id = imagem %>%
      str_to_lower() %>%                  # coloca tudo em minúsculas
      str_remove("-pan\\.jpg$")           # remove o final "-pan.jpg"
  )

head(dados)

imagem,obturado,ausente,cariado,CPOD,image_id
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ALP.025868-pan.jpg,10,8,0,18,alp.025868
ALP.025874-pan.jpg,5,15,0,20,alp.025874
ALP.025905-pan.jpg,7,11,0,18,alp.025905
ALP.025906-pan.jpg,11,5,0,16,alp.025906
ALP.025920-pan.jpg,3,3,0,6,alp.025920
ALP.025921-pan.jpg,2,0,0,2,alp.025921


In [19]:
resumo_clinica <- dados %>%
  group_by(clinica) %>%
  summarise(
    n_imagens = n(),
    cariado_mean   = mean(cariado, na.rm = TRUE),
    cariado_sd     = sd(cariado, na.rm = TRUE),
    ausente_mean   = mean(ausente, na.rm = TRUE),
    ausente_sd     = sd(ausente, na.rm = TRUE),
    obturado_mean  = mean(obturado, na.rm = TRUE),
    obturado_sd    = sd(obturado, na.rm = TRUE),
    cpod_mean      = mean(CPOD, na.rm = TRUE),
    cpod_sd        = sd(CPOD, na.rm = TRUE)
  )

resumo_clinica

ERROR: [1m[33mError[39m in `group_by()`:[22m
[1m[22m[33m![39m Must group by variables found in `.data`.
[31m✖[39m Column `clinica` is not found.


In [22]:
infos = read_csv("/home/aninha/Desktop/Doutorado/Dados/papiron_todas_pans/cpod_9_dez/resultados_com_origem_idade_sexo.csv")
glimpse(infos)

Rows: 192589 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): cd_exame, origem, sexo
dbl (1): idade

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 192,589
Columns: 4
$ cd_exame <chr> "alp.025868", "alp.025874", "alp.025905", "alp.025906", "alp.…
$ origem   <chr> "Alphaville", "Alphaville", "Alphaville", "Alphaville", "Alph…
$ idade    <dbl> 71, 57, 70, 52, 24, 32, 15, 34, 24, 31, 35, 28, 31, 28, 61, 1…
$ sexo     <chr> "feminino", "feminino", "masculino", "masculino", "feminino",…


In [23]:
glimpse(dados)

Rows: 196,670
Columns: 6
$ imagem   <chr> "ALP.025868-pan.jpg", "ALP.025874-pan.jpg", "ALP.025905-pan.j…
$ obturado <dbl> 10, 5, 7, 11, 3, 2, 5, 0, 11, 1, 11, 7, 6, 7, 15, 0, 13, 3, 3…
$ ausente  <dbl> 8, 15, 11, 5, 3, 0, 0, 4, 2, 0, 1, 0, 0, 0, 3, 2, 14, 3, 2, 8…
$ cariado  <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0…
$ CPOD     <dbl> 18, 20, 18, 16, 6, 2, 5, 4, 13, 1, 12, 7, 7, 7, 18, 2, 27, 6,…
$ image_id <chr> "alp.025868", "alp.025874", "alp.025905", "alp.025906", "alp.…


In [24]:
dados = merge(dados,infos,by.x = "image_id", by.y = "cd_exame")
glimpse(dados)

Rows: 192,589
Columns: 9
$ image_id <chr> "alp.025854", "alp.025868", "alp.025874", "alp.025876", "alp.…
$ imagem   <chr> "ALP.025854-pan.jpg", "ALP.025868-pan.jpg", "ALP.025874-pan.j…
$ obturado <dbl> 7, 10, 5, 12, 9, 13, 5, 15, 1, 2, 9, 6, 8, 5, 10, 7, 10, 7, 1…
$ ausente  <dbl> 4, 8, 15, 6, 4, 4, 0, 0, 6, 3, 1, 0, 5, 3, 3, 11, 2, 11, 5, 4…
$ cariado  <dbl> 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0…
$ CPOD     <dbl> 11, 18, 20, 18, 13, 17, 5, 16, 7, 5, 11, 6, 13, 8, 13, 18, 12…
$ origem   <chr> "Alphaville", "Alphaville", "Alphaville", "Alphaville", "Alph…
$ idade    <dbl> 29, 71, 57, 63, 31, 70, 40, 46, 32, 45, 27, 32, 42, 42, 45, 5…
$ sexo     <chr> "feminino", "feminino", "feminino", "feminino", "feminino", "…


In [25]:
# Cria e insere a faixa etária após 'idade'
add_faixa_etaria_oms <- function(df) {
  ordem_niveis <- c("5", "6-11", "12", "13-14", "15-19",
                    "20-34", "35-44", "45-64", "65-74", "75+")
  
  df %>%
    mutate(
      faixa_etaria_oms = case_when(
        is.na(idade)                ~ NA_character_,
        idade == 5                  ~ "5",
        idade >= 6  & idade <= 11   ~ "6-11",
        idade == 12                 ~ "12",
        idade >= 13 & idade <= 14   ~ "13-14",
        idade >= 15 & idade <= 19   ~ "15-19",
        idade >= 20 & idade <= 34   ~ "20-34",
        idade >= 35 & idade <= 44   ~ "35-44",
        idade >= 45 & idade <= 64   ~ "45-64",
        idade >= 65 & idade <= 74   ~ "65-74",
        idade >= 75                 ~ "75+",
        TRUE                        ~ NA_character_
      ),
      faixa_etaria_oms = factor(faixa_etaria_oms, levels = ordem_niveis, ordered = TRUE)
    ) %>%
    relocate(faixa_etaria_oms, .after = idade)
  }

In [26]:
dados = add_faixa_etaria_oms(dados)
glimpse(dados)

Rows: 192,589
Columns: 10
$ image_id         <chr> "alp.025854", "alp.025868", "alp.025874", "alp.025876…
$ imagem           <chr> "ALP.025854-pan.jpg", "ALP.025868-pan.jpg", "ALP.0258…
$ obturado         <dbl> 7, 10, 5, 12, 9, 13, 5, 15, 1, 2, 9, 6, 8, 5, 10, 7, …
$ ausente          <dbl> 4, 8, 15, 6, 4, 4, 0, 0, 6, 3, 1, 0, 5, 3, 3, 11, 2, …
$ cariado          <dbl> 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,…
$ CPOD             <dbl> 11, 18, 20, 18, 13, 17, 5, 16, 7, 5, 11, 6, 13, 8, 13…
$ origem           <chr> "Alphaville", "Alphaville", "Alphaville", "Alphaville…
$ idade            <dbl> 29, 71, 57, 63, 31, 70, 40, 46, 32, 45, 27, 32, 42, 4…
$ faixa_etaria_oms <ord> 20-34, 65-74, 45-64, 45-64, 20-34, 65-74, 35-44, 45-6…
$ sexo             <chr> "feminino", "feminino", "feminino", "feminino", "femi…


In [ ]:
glimpse(dados)

Rows: 192,589
Columns: 11
$ image_id <chr> "alp.025854", "alp.025868", "alp.025874", "alp.025876", "alp.…
$ imagem   <chr> "ALP.025854-pan.jpg", "ALP.025868-pan.jpg", "ALP.025874-pan.j…
$ obturado <dbl> 7, 10, 5, 12, 9, 13, 5, 15, 1, 2, 9, 6, 8, 5, 10, 7, 10, 7, 1…
$ ausente  <dbl> 4, 8, 15, 6, 4, 4, 0, 0, 6, 3, 1, 0, 5, 3, 3, 11, 2, 11, 5, 4…
$ cariado  <dbl> 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0…
$ CPOD     <dbl> 11, 18, 20, 18, 13, 17, 5, 16, 7, 5, 11, 6, 13, 8, 13, 18, 12…
$ prefix   <chr> "alp", "alp", "alp", "alp", "alp", "alp", "alp", "alp", "alp"…
$ clinica  <chr> "Alphaville", "Alphaville", "Alphaville", "Alphaville", "Alph…
$ origem   <chr> "Alphaville", "Alphaville", "Alphaville", "Alphaville", "Alph…
$ idade    <dbl> 29, 71, 57, 63, 31, 70, 40, 46, 32, 45, 27, 32, 42, 42, 45, 5…
$ sexo     <chr> "feminino", "feminino", "feminino", "feminino", "feminino", "…


In [27]:
library(dplyr)

# 1. Tabela de CARIADOS (C)
tabela_cariado <- dados %>%
  group_by(faixa_etaria_oms) %>%
  summarise(
    n = n(),
    mean_cariado = mean(cariado, na.rm = TRUE),
    median_cariado = median(cariado, na.rm = TRUE),
    sd_cariado = sd(cariado, na.rm = TRUE)
  )

# 2. Tabela de PERDIDOS (P)
tabela_ausente <- dados %>%
  group_by(faixa_etaria_oms) %>%
  summarise(
    n = n(),
    mean_ausente = mean(ausente, na.rm = TRUE),
    median_ausente = median(ausente, na.rm = TRUE),
    sd_ausente = sd(ausente, na.rm = TRUE)
  )

# 3. Tabela de OBTURADOS (O)
tabela_obturado <- dados %>%
  group_by(faixa_etaria_oms) %>%
  summarise(
    n = n(),
    mean_obturado = mean(obturado, na.rm = TRUE),
    median_obturado = median(obturado, na.rm = TRUE),
    sd_obturado = sd(obturado, na.rm = TRUE)
  )

# 4. Tabela de CPOD TOTAL
tabela_cpod <- dados %>%
  group_by(faixa_etaria_oms) %>%
  summarise(
    n = n(),
    mean_cpod = mean(CPOD, na.rm = TRUE),
    median_cpod = median(CPOD, na.rm = TRUE),
    sd_cpod = sd(CPOD, na.rm = TRUE)
  )


In [28]:
tabela_cpod

faixa_etaria_oms,n,mean_cpod,median_cpod,sd_cpod
<ord>,<int>,<dbl>,<dbl>,<dbl>
5,1727,9.266937,9.0,3.798056
6-11,18280,6.520077,6.0,3.251234
12,1642,2.689403,2.0,1.828676
13-14,2592,2.528935,2.0,1.922247
15-19,8623,3.351270,3.0,2.582685
20-34,50391,7.272727,6.0,4.749437
35-44,38087,12.092131,12.0,5.371651
45-64,52043,17.723613,18.0,5.131192
65-74,12563,20.055082,20.0,4.987326


In [29]:
tabela_cariado

faixa_etaria_oms,n,mean_cariado,median_cariado,sd_cariado
<ord>,<int>,<dbl>,<dbl>,<dbl>
5,1727,0.03474233,0,0.2041224
6-11,18280,0.03457330,0,0.2105263
12,1642,0.04323995,0,0.2288327
13-14,2592,0.06828704,0,0.2866603
15-19,8623,0.06157950,0,0.2821327
20-34,50391,0.09436209,0,0.3489078
35-44,38087,0.07327960,0,0.2921399
45-64,52043,0.03598947,0,0.2009542
65-74,12563,0.01456658,0,0.1269128


In [30]:
tabela_obturado

faixa_etaria_oms,n,mean_obturado,median_obturado,sd_obturado
<ord>,<int>,<dbl>,<dbl>,<dbl>
5,1727,0.2756225,0,0.6743108
6-11,18280,0.3858315,0,0.8575978
12,1642,0.6419001,0,1.0938411
13-14,2592,0.9185957,0,1.3706725
15-19,8623,1.8790444,1,2.2948118
20-34,50391,4.6584311,4,4.0675457
35-44,38087,8.2233571,8,4.6093148
45-64,52043,10.9728302,11,4.8626016
65-74,12563,9.5259890,10,5.3330434


In [31]:
tabela_ausente

faixa_etaria_oms,n,mean_ausente,median_ausente,sd_ausente
<ord>,<int>,<dbl>,<dbl>,<dbl>
5,1727,8.956572,9,3.774936
6-11,18280,6.099672,6,3.241745
12,1642,2.004263,2,1.848789
13-14,2592,1.542052,1,1.803489
15-19,8623,1.410646,1,1.748277
20-34,50391,2.519934,2,2.223821
35-44,38087,3.795495,4,2.951169
45-64,52043,6.714794,5,5.578786
65-74,12563,10.514527,8,7.436994


In [49]:
final = cbind(tabela_cpod,tabela_cariado,tabela_ausente,tabela_obturado)
final

faixa_etaria_oms,n,mean_cpod,median_cpod,sd_cpod,faixa_etaria_oms,n,mean_cariado,median_cariado,sd_cariado,faixa_etaria_oms,n,mean_ausente,median_ausente,sd_ausente,faixa_etaria_oms,n,mean_obturado,median_obturado,sd_obturado
<ord>,<int>,<dbl>,<dbl>,<dbl>,<ord>,<int>,<dbl>,<dbl>,<dbl>,<ord>,<int>,<dbl>,<dbl>,<dbl>,<ord>,<int>,<dbl>,<dbl>,<dbl>
5,1727,9.266937,9.0,3.798056,5,1727,0.03474233,0,0.2041224,5,1727,8.956572,9,3.774936,5,1727,0.2756225,0,0.6743108
6-11,18280,6.520077,6.0,3.251234,6-11,18280,0.03457330,0,0.2105263,6-11,18280,6.099672,6,3.241745,6-11,18280,0.3858315,0,0.8575978
12,1642,2.689403,2.0,1.828676,12,1642,0.04323995,0,0.2288327,12,1642,2.004263,2,1.848789,12,1642,0.6419001,0,1.0938411
13-14,2592,2.528935,2.0,1.922247,13-14,2592,0.06828704,0,0.2866603,13-14,2592,1.542052,1,1.803489,13-14,2592,0.9185957,0,1.3706725
15-19,8623,3.351270,3.0,2.582685,15-19,8623,0.06157950,0,0.2821327,15-19,8623,1.410646,1,1.748277,15-19,8623,1.8790444,1,2.2948118
20-34,50391,7.272727,6.0,4.749437,20-34,50391,0.09436209,0,0.3489078,20-34,50391,2.519934,2,2.223821,20-34,50391,4.6584311,4,4.0675457
35-44,38087,12.092131,12.0,5.371651,35-44,38087,0.07327960,0,0.2921399,35-44,38087,3.795495,4,2.951169,35-44,38087,8.2233571,8,4.6093148
45-64,52043,17.723613,18.0,5.131192,45-64,52043,0.03598947,0,0.2009542,45-64,52043,6.714794,5,5.578786,45-64,52043,10.9728302,11,4.8626016
65-74,12563,20.055082,20.0,4.987326,65-74,12563,0.01456658,0,0.1269128,65-74,12563,10.514527,8,7.436994,65-74,12563,9.5259890,10,5.3330434


In [57]:
final = final[-11,-c(2,4,6,7,9,11,12,14,16,17,19)]
final

,faixa_etaria_oms,mean_cpod,sd_cpod,mean_cariado,sd_cariado,mean_ausente,sd_ausente,mean_obturado,sd_obturado
,<ord>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,5,9.266937,3.798056,0.03474233,0.2041224,8.956572,3.774936,0.2756225,0.6743108
2,6-11,6.520077,3.251234,0.03457330,0.2105263,6.099672,3.241745,0.3858315,0.8575978
3,12,2.689403,1.828676,0.04323995,0.2288327,2.004263,1.848789,0.6419001,1.0938411
4,13-14,2.528935,1.922247,0.06828704,0.2866603,1.542052,1.803489,0.9185957,1.3706725
5,15-19,3.351270,2.582685,0.06157950,0.2821327,1.410646,1.748277,1.8790444,2.2948118
6,20-34,7.272727,4.749437,0.09436209,0.3489078,2.519934,2.223821,4.6584311,4.0675457
7,35-44,12.092131,5.371651,0.07327960,0.2921399,3.795495,2.951169,8.2233571,4.6093148
8,45-64,17.723613,5.131192,0.03598947,0.2009542,6.714794,5.578786,10.9728302,4.8626016
9,65-74,20.055082,4.987326,0.01456658,0.1269128,10.514527,7.436994,9.5259890,5.3330434


In [58]:
tabela_resumo_latex <- function(tab,
                                caption = "Summary table.",
                                label   = "tab:summary",
                                digits  = 2) {
  # garante data.frame
  tab <- as.data.frame(tab)
  
  # nomes das colunas
  cols <- names(tab)
  group_col <- cols[1]
  value_cols <- cols[-1]
  
  # formatação numérica
  fmt_num <- function(x, d = digits) formatC(x, format = "f", digits = d)
  
  # monta linhas da tabela
  linhas <- lapply(seq_len(nrow(tab)), function(i) {
    row <- tab[i, ]
    
    grupo <- as.character(row[[group_col]])
    
    valores <- vapply(
      value_cols,
      function(nm) {
        x <- row[[nm]]
        if (is.numeric(x)) {
          fmt_num(x)
        } else {
          as.character(x)
        }
      },
      FUN.VALUE = character(1)
    )
    
    paste0(
      grupo, " & ",
      paste(valores, collapse = " & "),
      " \\\\"
    )
  })
  
  corpo <- paste(unlist(linhas), collapse = "\n")
  
  # cabeçalho LaTeX (usa os nomes das colunas)
  header <- paste(cols, collapse = " & ")
  
  # monta LaTeX completo
  latex <- paste0(
"\\begin{table}[ht]\n",
"\\centering\n",
"\\caption{", caption, "}\n",
"\\label{", label, "}\n",
"\\begin{tabular}{", paste(rep("r", length(cols)), collapse = ""), "}\n",
"\\hline\n",
header, " \\\\\n",
"\\hline\n",
corpo, "\n",
"\\hline\n",
"\\end{tabular}\n",
"\\end{table}\n"
  )
  
  return(latex)
}


In [59]:
latex_code <- tabela_resumo_latex(final)
cat(latex_code)

\begin{table}[ht]
\centering
\caption{Summary table.}
\label{tab:summary}
\begin{tabular}{rrrrrrrrr}
\hline
faixa_etaria_oms & mean_cpod & sd_cpod & mean_cariado & sd_cariado & mean_ausente & sd_ausente & mean_obturado & sd_obturado \\
\hline
5 & 9.27 & 3.80 & 0.03 & 0.20 & 8.96 & 3.77 & 0.28 & 0.67 \\
6-11 & 6.52 & 3.25 & 0.03 & 0.21 & 6.10 & 3.24 & 0.39 & 0.86 \\
12 & 2.69 & 1.83 & 0.04 & 0.23 & 2.00 & 1.85 & 0.64 & 1.09 \\
13-14 & 2.53 & 1.92 & 0.07 & 0.29 & 1.54 & 1.80 & 0.92 & 1.37 \\
15-19 & 3.35 & 2.58 & 0.06 & 0.28 & 1.41 & 1.75 & 1.88 & 2.29 \\
20-34 & 7.27 & 4.75 & 0.09 & 0.35 & 2.52 & 2.22 & 4.66 & 4.07 \\
35-44 & 12.09 & 5.37 & 0.07 & 0.29 & 3.80 & 2.95 & 8.22 & 4.61 \\
45-64 & 17.72 & 5.13 & 0.04 & 0.20 & 6.71 & 5.58 & 10.97 & 4.86 \\
65-74 & 20.06 & 4.99 & 0.01 & 0.13 & 10.51 & 7.44 & 9.53 & 5.33 \\
75+ & 21.79 & 5.41 & 0.02 & 0.13 & 13.99 & 8.46 & 7.79 & 5.33 \\
\hline
\end{tabular}
\end{table}
